In [1]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Project directory:", Path.cwd())
print("Python version:", sys.version)

Python executable: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets\.venv\Scripts\python.exe
Project directory: C:\Users\vasan\Desktop\Capstone\Airport_Baggage_Baseline_Datasets
Python version: 3.11.6 (tags/v3.11.6:8b6ee5b, Oct  2 2023, 14:57:12) [MSC v.1935 64 bit (AMD64)]


In [2]:
import pandas as pd
import sqlalchemy
import psycopg2
import numpy as np

print("Pandas:", pd.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("PostgreSQL driver:", psycopg2.__version__)
print("NumPy:", np.__version__)

Pandas: 3.1.1
SQLAlchemy: 2.1.8
PostgreSQL driver: 2.9.13 (dt dec pq3 ext lo64)
NumPy: 2.4.6


In [3]:
from sqlalchemy import create_engine, text
from getpass import getpass
from urllib.parse import quote_plus

# Database configuration
DB_HOST = "127.0.0.1"
DB_PORT = "5433"
DB_NAME = "airport_baggage_db"
DB_USER = "postgres"

# Enter password securely
DB_PASSWORD = quote_plus(getpass("Enter PostgreSQL password: "))

# Create SQLAlchemy engine
DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("PostgreSQL engine created successfully.")

Enter PostgreSQL password:  ········


PostgreSQL engine created successfully.


In [4]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT current_database(), current_user, version();")
    )

    database_info = result.fetchone()

    print("Database:", database_info[0])
    print("User:", database_info[1])
    print("Version:", database_info[2])

Database: airport_baggage_db
User: postgres
Version: PostgreSQL 18.1 on x86_64-windows, compiled by msvc-19.44.35221, 64-bit


In [5]:
from pathlib import Path

project_path = Path.cwd()
processed_path = project_path / "data" / "processed"

csv_files = [
    "baggage_clean.csv",
    "flight_clean.csv",
    "baggage_assignment_clean.csv",
    "baggage_scan_event_clean.csv"
]

for file_name in csv_files:
    file_path = processed_path / file_name

    print(
        file_name,
        "->",
        "Found" if file_path.exists() else "Not Found"
    )

baggage_clean.csv -> Found
flight_clean.csv -> Found
baggage_assignment_clean.csv -> Found
baggage_scan_event_clean.csv -> Found


In [6]:
import pandas as pd

# Read cleaned CSV datasets
baggage_df = pd.read_csv(processed_path / "baggage_clean.csv")
flight_df = pd.read_csv(processed_path / "flight_clean.csv")
assignment_df = pd.read_csv(processed_path / "baggage_assignment_clean.csv")
scan_event_df = pd.read_csv(processed_path / "baggage_scan_event_clean.csv")

# Display dataset information
datasets = {
    "baggage": baggage_df,
    "flight": flight_df,
    "baggage_assignment": assignment_df,
    "baggage_scan_event": scan_event_df
}

for table_name, df in datasets.items():
    print(f"\nTable: {table_name}")
    print("Rows:", len(df))
    print("Columns:", list(df.columns))


Table: baggage
Rows: 1000
Columns: ['bag_id', 'tag_number', 'booking_reference', 'passenger_reference', 'created_at']

Table: flight
Rows: 250
Columns: ['flight_id', 'flight_number', 'source', 'destination', 'departure_time', 'arrival_time']

Table: baggage_assignment
Rows: 1500
Columns: ['assignment_id', 'bag_id', 'flight_id', 'segment_sequence', 'assigned_at']

Table: baggage_scan_event
Rows: 9000
Columns: ['event_id', 'bag_id', 'event_type', 'airport_code', 'terminal', 'flight_number', 'event_time', 'received_at', 'source_system']


In [7]:
for table_name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"TABLE: {table_name}")
    print(df.dtypes)


TABLE: baggage
bag_id                 str
tag_number             str
booking_reference      str
passenger_reference    str
created_at             str
dtype: object

TABLE: flight
flight_id         str
flight_number     str
source            str
destination       str
departure_time    str
arrival_time      str
dtype: object

TABLE: baggage_assignment
assignment_id         str
bag_id                str
flight_id             str
segment_sequence    int64
assigned_at           str
dtype: object

TABLE: baggage_scan_event
event_id         str
bag_id           str
event_type       str
airport_code     str
terminal         str
flight_number    str
event_time       str
received_at      str
source_system    str
dtype: object


In [8]:
# Define timestamp columns for each dataset
timestamp_columns = {
    "baggage": ["created_at"],
    "flight": ["departure_time", "arrival_time"],
    "baggage_assignment": ["assigned_at"],
    "baggage_scan_event": ["event_time", "received_at"]
}

# Convert timestamps to UTC-aware datetime
for table_name, columns in timestamp_columns.items():
    for column in columns:
        datasets[table_name][column] = pd.to_datetime(
            datasets[table_name][column],
            utc=True,
            errors="raise"
        )

print("Timestamp conversion completed successfully.")

Timestamp conversion completed successfully.


In [9]:
for table_name, df in datasets.items():
    print(f"\n{table_name}")
    print(df.dtypes)


baggage
bag_id                                 str
tag_number                             str
booking_reference                      str
passenger_reference                    str
created_at             datetime64[us, UTC]
dtype: object

flight
flight_id                         str
flight_number                     str
source                            str
destination                       str
departure_time    datetime64[us, UTC]
arrival_time      datetime64[us, UTC]
dtype: object

baggage_assignment
assignment_id                       str
bag_id                              str
flight_id                           str
segment_sequence                  int64
assigned_at         datetime64[us, UTC]
dtype: object

baggage_scan_event
event_id                         str
bag_id                           str
event_type                       str
airport_code                     str
terminal                         str
flight_number                    str
event_time       datetime64[us, UTC]

In [10]:
baggage_df = datasets["baggage"]

baggage_df.to_sql(
    name="baggage",
    con=engine,
    schema="public",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=500
)

print("Baggage data loaded successfully!")
print("Records loaded:", len(baggage_df))

Baggage data loaded successfully!
Records loaded: 1000


In [11]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM baggage;")
    )

    baggage_count = result.scalar()

print("Baggage records in PostgreSQL:", baggage_count)

Baggage records in PostgreSQL: 1000


In [12]:
with engine.connect() as connection:
    sample_baggage = pd.read_sql(
        text("SELECT * FROM baggage LIMIT 5;"),
        connection
    )

display(sample_baggage)

,bag_id,tag_number,booking_reference,passenger_reference,created_at
0,BAG00001,TAG63643924,PNR812526,PAX00001,2026-11-06 04:50:00+00:00
1,BAG00002,TAG10226999,PNR645175,PAX00002,2026-11-06 06:03:00+00:00
2,BAG00003,TAG27084279,PNR301158,PAX00003,2026-11-03 13:37:00+00:00
3,BAG00004,TAG32775593,PNR790855,PAX00004,2026-10-26 22:18:00+00:00
4,BAG00005,TAG76354180,PNR519066,PAX00005,2026-10-30 13:14:00+00:00


In [13]:
flight_df = datasets["flight"]

flight_df.to_sql(
    name="flight",
    con=engine,
    schema="public",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=500
)

print("Flight data loaded successfully!")
print("Records loaded:", len(flight_df))

Flight data loaded successfully!
Records loaded: 250


In [14]:
with engine.connect() as connection:
    counts = connection.execute(
        text("""
            SELECT
                (SELECT COUNT(*) FROM baggage) AS baggage_count,
                (SELECT COUNT(*) FROM flight) AS flight_count;
        """)
    ).fetchone()

print("Baggage records:", counts[0])
print("Flight records:", counts[1])

Baggage records: 1000
Flight records: 250


In [15]:
assignment_df = datasets["baggage_assignment"]

assignment_df.to_sql(
    name="baggage_assignment",
    con=engine,
    schema="public",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=500
)

print("Baggage assignment data loaded successfully!")
print("Records loaded:", len(assignment_df))

Baggage assignment data loaded successfully!
Records loaded: 1500


In [16]:
scan_event_df = datasets["baggage_scan_event"]

scan_event_df.to_sql(
    name="baggage_scan_event",
    con=engine,
    schema="public",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=500
)

print("Baggage scan events loaded successfully!")
print("Records loaded:", len(scan_event_df))

Baggage scan events loaded successfully!
Records loaded: 9000


In [17]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT 'baggage' AS table_name, COUNT(*) AS total_records
        FROM baggage

        UNION ALL

        SELECT 'flight', COUNT(*) FROM flight

        UNION ALL

        SELECT 'baggage_assignment', COUNT(*) FROM baggage_assignment

        UNION ALL

        SELECT 'baggage_scan_event', COUNT(*) FROM baggage_scan_event;
    """))

    validation_df = pd.DataFrame(
        result.fetchall(),
        columns=result.keys()
    )

display(validation_df)

,table_name,total_records
0,baggage,1000
1,flight,250
2,baggage_assignment,1500
3,baggage_scan_event,9000


In [18]:
pk_validation_query = text("""
    SELECT
        'baggage' AS table_name,
        COUNT(*) AS total_records,
        COUNT(DISTINCT bag_id) AS unique_ids
    FROM baggage

    UNION ALL

    SELECT
        'flight',
        COUNT(*),
        COUNT(DISTINCT flight_id)
    FROM flight

    UNION ALL

    SELECT
        'baggage_assignment',
        COUNT(*),
        COUNT(DISTINCT assignment_id)
    FROM baggage_assignment

    UNION ALL

    SELECT
        'baggage_scan_event',
        COUNT(*),
        COUNT(DISTINCT event_id)
    FROM baggage_scan_event;
""")

with engine.connect() as connection:
    pk_validation_df = pd.read_sql(
        pk_validation_query,
        connection
    )

pk_validation_df["duplicate_ids"] = (
    pk_validation_df["total_records"]
    - pk_validation_df["unique_ids"]
)

display(pk_validation_df)

,table_name,total_records,unique_ids,duplicate_ids
0,baggage,1000,1000,0
1,flight,250,250,0
2,baggage_assignment,1500,1500,0
3,baggage_scan_event,9000,9000,0


In [19]:
fk_validation_query = text("""
    SELECT
        'assignment_bag_id' AS relationship,
        COUNT(*) AS total_records,
        COUNT(b.bag_id) AS matched_records,
        COUNT(*) - COUNT(b.bag_id) AS invalid_records
    FROM baggage_assignment a
    LEFT JOIN baggage b
        ON a.bag_id = b.bag_id

    UNION ALL

    SELECT
        'assignment_flight_id',
        COUNT(*),
        COUNT(f.flight_id),
        COUNT(*) - COUNT(f.flight_id)
    FROM baggage_assignment a
    LEFT JOIN flight f
        ON a.flight_id = f.flight_id

    UNION ALL

    SELECT
        'scan_event_bag_id',
        COUNT(*),
        COUNT(b.bag_id),
        COUNT(*) - COUNT(b.bag_id)
    FROM baggage_scan_event s
    LEFT JOIN baggage b
        ON s.bag_id = b.bag_id;
""")

with engine.connect() as connection:
    fk_validation_df = pd.read_sql(
        fk_validation_query,
        connection
    )

display(fk_validation_df)

,relationship,total_records,matched_records,invalid_records
0,assignment_bag_id,1500,1500,0
1,assignment_flight_id,1500,1500,0
2,scan_event_bag_id,9000,9000,0


In [20]:
null_validation_query = text("""
    SELECT
        COUNT(*) AS total_events,
        COUNT(flight_number) AS non_null_flight_numbers,
        COUNT(*) - COUNT(flight_number) AS null_flight_numbers
    FROM baggage_scan_event;
""")

with engine.connect() as connection:
    null_validation_df = pd.read_sql(
        null_validation_query,
        connection
    )

display(null_validation_df)

,total_events,non_null_flight_numbers,null_flight_numbers
0,9000,5500,3500


In [21]:
journey_type_query = text("""
    SELECT
        CASE
            WHEN COUNT(*) = 1 THEN 'DIRECT'
            WHEN COUNT(*) = 2 THEN 'CONNECTING'
            ELSE 'OTHER'
        END AS journey_type,
        COUNT(*) AS baggage_count
    FROM baggage_assignment
    GROUP BY bag_id
    ORDER BY journey_type;
""")

with engine.connect() as connection:
    journey_type_df = pd.read_sql(
        journey_type_query,
        connection
    )

display(journey_type_df)

,journey_type,baggage_count
0,CONNECTING,2
1,CONNECTING,2
2,CONNECTING,2
3,CONNECTING,2
4,CONNECTING,2
...,...,...
995,DIRECT,1
996,DIRECT,1
997,DIRECT,1
998,DIRECT,1


In [22]:
journey_type_query = text("""
    SELECT
        journey_type,
        COUNT(*) AS baggage_count
    FROM (
        SELECT
            bag_id,
            CASE
                WHEN COUNT(*) = 1 THEN 'DIRECT'
                WHEN COUNT(*) = 2 THEN 'CONNECTING'
                ELSE 'OTHER'
            END AS journey_type
        FROM baggage_assignment
        GROUP BY bag_id
    ) AS baggage_journeys
    GROUP BY journey_type
    ORDER BY journey_type;
""")

with engine.connect() as connection:
    journey_type_df = pd.read_sql(
        journey_type_query,
        connection
    )

display(journey_type_df)

,journey_type,baggage_count
0,CONNECTING,500
1,DIRECT,500


In [23]:
timing_investigation_query = text("""
    WITH flight_segments AS (
        SELECT
            ba.bag_id,
            ba.segment_sequence,
            f.departure_time,
            f.arrival_time
        FROM baggage_assignment ba
        JOIN flight f
            ON ba.flight_id = f.flight_id
    ),
    connecting_journeys AS (
        SELECT
            first_segment.bag_id,
            first_segment.arrival_time AS first_arrival,
            second_segment.departure_time AS second_departure
        FROM flight_segments first_segment
        JOIN flight_segments second_segment
            ON first_segment.bag_id = second_segment.bag_id
        WHERE first_segment.segment_sequence = 1
          AND second_segment.segment_sequence = 2
    )
    SELECT
        bag_id,
        first_arrival,
        second_departure,
        ROUND(
            (EXTRACT(EPOCH FROM
                (second_departure - first_arrival)) / 60)::numeric,
            2
        ) AS connection_gap_minutes
    FROM connecting_journeys
    WHERE second_departure < first_arrival
    ORDER BY bag_id;
""")

with engine.connect() as connection:
    timing_investigation_df = pd.read_sql(
        timing_investigation_query,
        connection
    )

print("Impossible connections:", len(timing_investigation_df))
display(timing_investigation_df.head(10))

Impossible connections: 185


,bag_id,first_arrival,second_departure,connection_gap_minutes
0,BAG00501,2026-11-01 18:00:00+00:00,2026-11-01 06:00:00+00:00,-720.0
1,BAG00506,2026-11-06 18:00:00+00:00,2026-11-06 06:00:00+00:00,-720.0
2,BAG00509,2026-11-09 18:00:00+00:00,2026-11-09 06:00:00+00:00,-720.0
3,BAG00510,2026-11-10 18:00:00+00:00,2026-11-10 06:00:00+00:00,-720.0
4,BAG00511,2026-11-01 18:00:00+00:00,2026-11-01 06:00:00+00:00,-720.0
5,BAG00513,2026-11-03 18:00:00+00:00,2026-11-03 06:00:00+00:00,-720.0
6,BAG00514,2026-11-04 18:00:00+00:00,2026-11-04 06:00:00+00:00,-720.0
7,BAG00521,2026-11-01 18:00:00+00:00,2026-11-01 06:00:00+00:00,-720.0
8,BAG00522,2026-11-02 18:00:00+00:00,2026-11-02 06:00:00+00:00,-720.0
9,BAG00525,2026-11-05 18:00:00+00:00,2026-11-05 06:00:00+00:00,-720.0


In [24]:
handoff_query = text("""
    WITH transfer_events AS (
        SELECT
            bag_id,
            event_time AS transfer_time
        FROM baggage_scan_event
        WHERE event_type = 'TRANSFER'
    ),
    second_flight_loaded AS (
        SELECT
            ba.bag_id,
            s.event_time AS loaded_time
        FROM baggage_assignment ba
        JOIN flight f
            ON ba.flight_id = f.flight_id
        JOIN baggage_scan_event s
            ON s.bag_id = ba.bag_id
           AND s.flight_number = f.flight_number
           AND s.event_type = 'LOADED'
        WHERE ba.segment_sequence = 2
    )
    SELECT
        t.bag_id,
        t.transfer_time,
        l.loaded_time,
        ROUND(
            (EXTRACT(EPOCH FROM
                (l.loaded_time - t.transfer_time)) / 60)::numeric,
            2
        ) AS handoff_gap_minutes
    FROM transfer_events t
    JOIN second_flight_loaded l
        ON t.bag_id = l.bag_id
    WHERE l.loaded_time < t.transfer_time
    ORDER BY t.bag_id;
""")

with engine.connect() as connection:
    handoff_df = pd.read_sql(handoff_query, connection)

print("Baggage loaded before transfer:", len(handoff_df))
display(handoff_df.head(10))

Baggage loaded before transfer: 185


,bag_id,transfer_time,loaded_time,handoff_gap_minutes
0,BAG00501,2026-11-01 18:45:00+00:00,2026-11-01 05:15:00+00:00,-810.0
1,BAG00506,2026-11-06 18:45:00+00:00,2026-11-06 05:15:00+00:00,-810.0
2,BAG00509,2026-11-09 18:45:00+00:00,2026-11-09 05:15:00+00:00,-810.0
3,BAG00510,2026-11-10 18:45:00+00:00,2026-11-10 05:15:00+00:00,-810.0
4,BAG00511,2026-11-01 18:45:00+00:00,2026-11-01 05:15:00+00:00,-810.0
5,BAG00513,2026-11-03 18:45:00+00:00,2026-11-03 05:15:00+00:00,-810.0
6,BAG00514,2026-11-04 18:45:00+00:00,2026-11-04 05:15:00+00:00,-810.0
7,BAG00521,2026-11-01 18:45:00+00:00,2026-11-01 05:15:00+00:00,-810.0
8,BAG00522,2026-11-02 18:45:00+00:00,2026-11-02 05:15:00+00:00,-810.0
9,BAG00525,2026-11-05 18:45:00+00:00,2026-11-05 05:15:00+00:00,-810.0


In [25]:
airport_journey_query = text("""
    WITH expected_airports AS (
        SELECT
            ba.bag_id,
            MIN(f.source) FILTER (
                WHERE ba.segment_sequence = 1
            ) AS origin_airport,
            MAX(f.destination) FILTER (
                WHERE ba.segment_sequence = 1
            ) AS first_destination,
            MAX(f.destination) FILTER (
                WHERE ba.segment_sequence = 2
            ) AS final_destination
        FROM baggage_assignment ba
        JOIN flight f
            ON ba.flight_id = f.flight_id
        GROUP BY ba.bag_id
    )
    SELECT
        journey_type,
        COUNT(*) AS baggage_count
    FROM (
        SELECT
            bag_id,
            CASE
                WHEN final_destination IS NULL
                    THEN 'DIRECT'
                ELSE 'CONNECTING'
            END AS journey_type
        FROM expected_airports
    ) journeys
    GROUP BY journey_type
    ORDER BY journey_type;
""")

with engine.connect() as connection:
    airport_journey_df = pd.read_sql(
        airport_journey_query,
        connection
    )

display(airport_journey_df)

,journey_type,baggage_count
0,CONNECTING,500
1,DIRECT,500


In [26]:
investigation_file = (
    processed_path / "final_baggage_misroute_investigation.csv"
)

print("File exists:", investigation_file.exists())

if investigation_file.exists():
    print("File size:", investigation_file.stat().st_size, "bytes")

File exists: True
File size: 86278 bytes


In [27]:
investigation_df = pd.read_csv(investigation_file)

print("Dataset shape:", investigation_df.shape)
print("\nColumn names:")
print(investigation_df.columns.tolist())

print("\nData types:")
display(investigation_df.dtypes.to_frame("data_type"))

print("\nSample records:")
display(investigation_df.head())

print("\nInvestigation status distribution:")
display(
    investigation_df["investigation_status"]
    .value_counts()
    .to_frame("baggage_count")
)

Dataset shape: (1000, 12)

Column names:
['bag_id', 'unexpected_airport_detected', 'incorrect_flight_detected', 'incorrect_airport_flight_detected', 'segment_event_investigation_required', 'connection_gap_minutes', 'impossible_connection', 'handoff_gap_minutes', 'loaded_before_transfer', 'journey_type', 'event_sequence_matches', 'investigation_status']

Data types:


,data_type
bag_id,str
unexpected_airport_detected,bool
incorrect_flight_detected,bool
incorrect_airport_flight_detected,bool
segment_event_investigation_required,bool
connection_gap_minutes,float64
impossible_connection,bool
handoff_gap_minutes,float64
loaded_before_transfer,bool
journey_type,str



Sample records:


,bag_id,unexpected_airport_detected,incorrect_flight_detected,incorrect_airport_flight_detected,segment_event_investigation_required,connection_gap_minutes,impossible_connection,handoff_gap_minutes,loaded_before_transfer,journey_type,event_sequence_matches,investigation_status
0,BAG00001,False,False,False,False,NaN,False,NaN,False,DIRECT,True,NO_ISSUE_DETECTED
1,BAG00002,False,False,False,False,NaN,False,NaN,False,DIRECT,True,NO_ISSUE_DETECTED
2,BAG00003,False,False,False,False,NaN,False,NaN,False,DIRECT,True,NO_ISSUE_DETECTED
3,BAG00004,False,False,False,False,NaN,False,NaN,False,DIRECT,True,NO_ISSUE_DETECTED
4,BAG00005,False,False,False,False,NaN,False,NaN,False,DIRECT,True,NO_ISSUE_DETECTED



Investigation status distribution:


,baggage_count
investigation_status,
NO_ISSUE_DETECTED,500
EVENT_PATTERN_REVIEW,315
TIMING_INCONSISTENCY,185


In [29]:
investigation_df.to_sql(
    name="baggage_investigation",
    con=engine,
    if_exists="append",
    index=False
)

print("Investigation records loaded successfully!")
print("Records loaded:", len(investigation_df))

Investigation records loaded successfully!
Records loaded: 1000


In [30]:
investigation_validation_query = text("""
    SELECT
        investigation_status,
        COUNT(*) AS baggage_count
    FROM baggage_investigation
    GROUP BY investigation_status
    ORDER BY investigation_status;
""")

with engine.connect() as connection:
    investigation_validation_df = pd.read_sql(
        investigation_validation_query,
        connection
    )

display(investigation_validation_df)

with engine.connect() as connection:
    total_records = connection.execute(
        text("SELECT COUNT(*) FROM baggage_investigation")
    ).scalar()

print("Total investigation records:", total_records)

,investigation_status,baggage_count
0,EVENT_PATTERN_REVIEW,315
1,NO_ISSUE_DETECTED,500
2,TIMING_INCONSISTENCY,185


Total investigation records: 1000


In [31]:
final_table_validation_query = text("""
    SELECT 'baggage' AS table_name, COUNT(*) AS record_count
    FROM baggage

    UNION ALL

    SELECT 'flight', COUNT(*) FROM flight

    UNION ALL

    SELECT 'baggage_assignment', COUNT(*) FROM baggage_assignment

    UNION ALL

    SELECT 'baggage_scan_event', COUNT(*) FROM baggage_scan_event

    UNION ALL

    SELECT 'baggage_investigation', COUNT(*) FROM baggage_investigation;
""")

with engine.connect() as connection:
    final_table_validation_df = pd.read_sql(
        final_table_validation_query,
        connection
    )

display(final_table_validation_df)

print(
    "Total records across all tables:",
    final_table_validation_df["record_count"].sum()
)

,table_name,record_count
0,baggage,1000
1,flight,250
2,baggage_assignment,1500
3,baggage_scan_event,9000
4,baggage_investigation,1000


Total records across all tables: 12750


In [32]:
%pip install snowflake-connector-python

   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.6 MB 16.7 MB/s eta 0:00:01
   ------- -------------------------------- 2.1/11.6 MB 5.3 MB/s eta 0:00:02
   ---------- ----------------------------- 3.1/11.6 MB 6.4 MB/s eta 0:00:02
   ------------------ --------------------- 5.2/11.6 MB 6.8 MB/s eta 0:00:01
   --------------------- ------------------ 6.3/11.6 MB 6.1 MB/s eta 0:00:01
   ----------------------- ---------------- 6.8/11.6 MB 5.3 MB/s eta 0:00:01
   ------------------------- -------------- 7.3/11.6 MB 5.6 MB/s eta 0:00:01
   -------------------------------- ------- 9.4/11.6 MB 5.9 MB/s eta 0:00:01
   ------------------------------------ --- 10.5/11.6 MB 5.5 MB/s eta 0:00:01
   -------------------------------------- - 11.3/11.6 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------  11.5/11.6 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 11.6/11.6 MB 4.7 MB/s  0:00:02
   -